In [19]:
from data_loader import load_mimic_tables                                      
                                                                                 
tables = load_mimic_tables()                                                                                                        
                                                                                 
# See all available tables:                                                    
print(tables.keys())  

/Users/huseyin/Codes/gpt-medic/explore/data_loader.py:26: DtypeWarning: Columns (0: administration_type, 1: barcode_type, 2: reason_for_no_barcode, 3: complete_dose_not_given, 4: dose_due, 5: dose_due_unit, 6: dose_given, 7: dose_given_unit, 8: will_remainder_of_dose_be_given, 9: product_unit, 10: product_code, 11: product_description, 12: product_description_other, 13: infusion_rate_adjustment, 14: infusion_rate_unit, 15: route, 16: infusion_complete, 17: completion_interval, 18: new_iv_bag_hung, 19: continued_infusion_in_other_location, 20: restart_interval, 21: side, 22: site, 23: non_formulary_visual_verification) have mixed types. Specify dtype option on import or set low_memory=False.
  tables[table_name] = pd.read_csv(file, compression="gzip")


dict_keys(['hosp.poe', 'hosp.d_hcpcs', 'hosp.poe_detail', 'hosp.patients', 'hosp.diagnoses_icd', 'hosp.emar_detail', 'hosp.provider', 'hosp.prescriptions', 'hosp.drgcodes', 'hosp.d_icd_diagnoses', 'hosp.d_labitems', 'hosp.transfers', 'hosp.admissions', 'hosp.labevents', 'hosp.pharmacy', 'hosp.procedures_icd', 'hosp.hcpcsevents', 'hosp.services', 'hosp.d_icd_procedures', 'hosp.omr', 'hosp.emar', 'hosp.microbiologyevents', 'icu.datetimeevents', 'icu.caregiver', 'icu.ingredientevents', 'icu.inputevents', 'icu.procedureevents', 'icu.d_items', 'icu.chartevents', 'icu.icustays', 'icu.outputevents'])


In [20]:
from tokenizer.flextpp import PatientTimelineTokenizer, Event

# Create and fit the tokenizer
tokenizer = PatientTimelineTokenizer()
tokenizer.fit(tables)

print("Tokenizer fitted successfully!")
print(f"Combined vocabulary size: {len(tokenizer.get_combined_vocabulary())}")

Tokenizer fitted successfully!
Combined vocabulary size: 2661


/Users/huseyin/Codes/gpt-medic/explore/tokenizer/flextpp/diagnosis.py:52: UserWarning: 731 ICD-9 diagnosis codes were missed during conversion to ICD-10. Examples: <ArrowStringArray>
['D696', 'S030XXA', 'S25512A', 'D649', 'F1010']
Length: 5, dtype: str
  warnings.warn(f"{len(missing_codes)} ICD-9 diagnosis codes were missed during conversion to ICD-10. Examples: {missing_codes[:5]}")
/Users/huseyin/Codes/gpt-medic/explore/tokenizer/flextpp/procedure.py:52: UserWarning: 169 ICD-9 procedure codes were missed during conversion to ICD-10. Examples: <ArrowStringArray>
['02H633Z', '0BH17EZ', 'B211YZZ', '3E0G76Z', '0BCH8ZZ']
Length: 5, dtype: str
  warnings.warn(f"{len(missing_codes)} ICD-9 procedure codes were missed during conversion to ICD-10. Examples: {missing_codes[:5]}")


In [21]:
# Get one example hospital admission
admissions = tables['hosp.admissions']
example_hadm_id = admissions['hadm_id'].iloc[0]

print(f"Example hadm_id: {example_hadm_id}")

# Tokenize the session
result = tokenizer.tokenize_session(example_hadm_id, tables)

print(f"\nPatient: {result['subject_id']}")
print(f"Admission: {result['admit_time']}")
print(f"Discharge: {result['discharge_time']}")
print(f"Total events: {len(result['events'])}")

Example hadm_id: 24181354

Patient: 10004235
Admission: 2196-02-24 14:38:00
Discharge: 2196-03-04 14:02:00
Total events: 875


In [22]:
# Show first 20 events
print("First 20 events:\n")
print(f"{'Time (days)':<12} {'Type':<6} {'Tokens'}")
print("-" * 80)

for event in result['events'][:-1]:
    print(f"{event.time:<12.4f} {event.event_type:<6} {event.tokens}")

First 20 events:

Time (days)  Type   Tokens
--------------------------------------------------------------------------------
-0.6097      PROC   <PROC> <3891>
-0.6097      PROC   <PROC> <9671>
0.0000       DEMO   <SEX> <M> <MARITAL> <SINGLE> <RACE> <BLACK_CAPE_VERDEAN>
0.0000       DIAG   <ICD> <03842>
0.0000       DIAG   <ICD> <78551>
0.0000       DIAG   <ICD> <5845>
0.0000       DIAG   <ICD> <570>
0.0000       DIAG   <ICD> <51881>
0.0000       DIAG   <ICD> <486>
0.0000       DIAG   <ICD> <34830>
0.0000       DIAG   <ICD> <5761>
0.0000       DIAG   <ICD> <42821>
0.0000       DIAG   <ICD> <29900>
0.0000       DIAG   <ICD> <2762>
0.0000       DIAG   <ICD> <75169>
0.0000       DIAG   <ICD> <4254>
0.0000       DIAG   <ICD> <99592>
0.0000       DIAG   <ICD> <42731>
0.0000       DIAG   <ICD> <7906>
0.0000       DIAG   <ICD> <28749>
0.0000       DIAG   <ICD> <79029>
0.0000       DIAG   <ICD> <V1253>
0.0000       DIAG   <ICD> <5939>
0.0000       DIAG   <ICD> <2749>
0.0000       DIAG   <ICD> 

In [23]:
# Show one event in detail
print("Example Event structure:\n")
event = result['events'][5] if len(result['events']) > 5 else result['events'][0]
print(f"Event(")
print(f"    time={event.time},")
print(f"    tokens='{event.tokens}',")
print(f"    event_type='{event.event_type}',")
print(f"    raw_data={event.raw_data}")
print(f")")

Example Event structure:

Event(
    time=0.0,
    tokens='<ICD> <5845>',
    event_type='DIAG',
    raw_data={'icd_code': '5845', 'seq_num': 3}
)
